# 🌾 Ultimate Intelligent Farm Recommendation System
## ML/DL Ensemble + HDBSCAN + SHAP + Optuna + Uncertainty + Yield + Multi-Objective + Vector RAG + Federated Design + Llama 3 + REST API

---

## 📌 What This Notebook Implements

| Module | Technique | Purpose |
|--------|-----------|---------|
| Clustering | HDBSCAN | Density-based clustering — no K needed, handles outliers |
| Explainability | SHAP | Per-farm feature contribution breakdown |
| Hyperparameter tuning | Optuna | Bayesian optimisation — finds best model params automatically |
| Calibration | Platt / Isotonic | Makes confidence % statistically honest |
| Uncertainty | Monte Carlo Dropout | Returns prediction intervals, not just point estimates |
| Yield prediction | Gradient Boosting Regressor | Quantitative tons/ha forecast alongside crop type |
| Satellite | Time-series NDVI | 12-month vegetation trajectory instead of single snapshot |
| RAG | ChromaDB + Sentence Transformers | Retrieve similar historical farms to enrich LLM prompt |
| Multi-objective | NSGA-II / Pareto | Jointly optimise suitability, yield, water use, carbon |
| Federated design | FedAvg architecture | Privacy-preserving multi-farm learning blueprint |
| LLM | Llama 3.1-8B via Groq | Natural language explanation + multi-turn conversation |
| Deployment | Hugging Face Spaces + Flask | Permanent public API — no ngrok timeouts |

---

## 🧠 Full System Architecture

```
Farm Sensors (N,P,K,pH,Temp,Hum,Rain)
        +
Sentinel-2 Time-Series NDVI (12 months)
        │
        ▼
Feature Engineering (18 features + temporal NDVI stats)
        │
        ├──► Optuna Hyperparameter Optimisation
        │
        ▼
Autoencoder → Latent Space (8D)
        │
        ▼
HDBSCAN Clustering (density-based, outlier-aware)
        │
        ▼
Per-Cluster Model Portfolio
   ├── RF  (Optuna-tuned, Calibrated)
   ├── XGB (Optuna-tuned, Calibrated)
   ├── GBM (Calibrated)
   ├── 1D-CNN (MC-Dropout uncertainty)
   └── TabNet-MLP (MC-Dropout uncertainty)
        │
        ├──► SHAP Explainability Layer
        ├──► Uncertainty Quantification (MC-Dropout)
        ├──► Yield Regression Model
        └──► Multi-Objective Pareto Optimisation
        │
        ▼
ChromaDB Vector RAG
(retrieve 5 most similar historical farms)
        │
        ▼
Llama 3.1-8B (Groq) — enriched with RAG context
        │
        ▼
Structured JSON Response
(crop, confidence interval, yield, SHAP explanation,
 Pareto-optimal alternatives, LLM narrative)
        │
        ▼
Flask REST API → Hugging Face Spaces (permanent URL)
        +
Federated Learning Architecture Design
```


---
# CELL 1 — Install All Dependencies

In [ ]:
# ============================================================
# CELL 1 — Install all required packages
#
# New packages vs previous version:
#   hdbscan      : density-based clustering (replaces KMeans)
#   shap         : SHapley Additive exPlanations for model interpretability
#   optuna       : Bayesian hyperparameter optimisation framework
#   chromadb     : local vector database for RAG (similar farm retrieval)
#   sentence-transformers : embed farm feature vectors for ChromaDB
#   groq         : Llama 3.1 free API client
#   flask/flask-cors : REST API server
#   pymoo        : multi-objective optimisation (NSGA-II / Pareto)
#   scikit-learn calibration: built into sklearn, no extra install
#   MC-Dropout uncertainty: built into Keras, no extra install
#
# DO NOT re-run this cell after the runtime restarts.
# ============================================================
import subprocess, sys

packages = [
    'xgboost',
    'plotly',
    'groq',
    'flask',
    'flask-cors',
    'pyngrok',
    'hdbscan',          # HDBSCAN clustering
    'shap',             # SHAP explainability
    'optuna',           # Bayesian hyperparameter search
    'chromadb',         # Vector database for RAG
    'sentence-transformers',  # Farm embedding for ChromaDB
    'pymoo',            # Multi-objective optimisation (NSGA-II)
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    print(f'  ✅ {pkg}')

import os
print('\n✅ All packages installed. Restarting runtime...')
os.kill(os.getpid(), 9)


---
# CELL 2 — Imports & Environment Verification

In [ ]:
# ============================================================
# CELL 2 — All imports
# Run this AFTER the runtime restarts from Cell 1.
# ============================================================
import os, warnings, json, time, re, threading
warnings.filterwarnings('ignore')
os.environ.update({'TF_CPP_MIN_LOG_LEVEL': '3', 'TF_ENABLE_ONEDNN_OPTS': '0'})

# ---- Core data science ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly

# ---- Sklearn ----
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    classification_report, confusion_matrix,
    accuracy_score, mean_squared_error, r2_score
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, GradientBoostingRegressor
)
# CalibratedClassifierCV wraps any classifier with Platt scaling or isotonic
# regression so that predict_proba() returns honest probabilities.
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import sklearn

# ---- XGBoost & HDBSCAN ----
from xgboost import XGBClassifier
import xgboost
import hdbscan     # density-based clustering — no K required

# ---- SHAP ----
import shap        # model-agnostic feature importance

# ---- Optuna ----
import optuna      # Bayesian hyperparameter optimisation
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---- TensorFlow / Keras ----
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ---- ChromaDB & Sentence Transformers (RAG) ----
import chromadb
from sentence_transformers import SentenceTransformer

# ---- Pymoo (multi-objective optimisation) ----
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.optimize import minimize as pymoo_minimize
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

# ---- Groq (Llama 3.1) ----
from groq import Groq

# ---- Flask (REST API) ----
from flask import Flask, request, jsonify
from flask_cors import CORS

# ---- Reproducibility ----
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Verification ----
print('=' * 50)
print('  ENVIRONMENT VERIFICATION')
print('=' * 50)
for name, version in [
    ('NumPy',         np.__version__),
    ('Pandas',        pd.__version__),
    ('TensorFlow',    tf.__version__),
    ('Scikit-learn',  sklearn.__version__),
    ('XGBoost',       xgboost.__version__),
    ('Seaborn',       sns.__version__),
    ('Plotly',        plotly.__version__),
]:
    print(f'  {name:<18}: {version}')

import groq as groq_pkg
print(f'  {"Groq":<18}: {groq_pkg.__version__}')
print('=' * 50)
print('✅ All libraries loaded.')


---
# CELL 3 — API Keys & Vector Database Setup

In [ ]:
# ============================================================
# CELL 3 — Configure Groq API key and initialise ChromaDB
#
# GROQ API KEY:
#   1. Go to https://console.groq.com
#   2. Sign up (free) — no credit card required
#   3. Create an API key
#   4. In Colab: click the 🔑 icon (left sidebar) → Add secret
#      named GROQ_API_KEY and paste your key
#
# CHROMADB:
#   Runs entirely locally inside Colab — no external service needed.
#   Stores farm feature vectors as embeddings so we can retrieve
#   the 5 most agronomically similar historical farms at inference
#   time and inject them into the Llama prompt (RAG pattern).
# ============================================================
from google.colab import userdata

# ---- Groq ----
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('✅ Groq key loaded from Colab Secrets.')
except Exception:
    GROQ_API_KEY = 'gsk_YOUR_KEY_HERE'   # fallback — replace with your key
    print('⚠️  Using hardcoded Groq key.')

groq_client = Groq(api_key=GROQ_API_KEY)

# Quick Groq connectivity test
try:
    test = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role':'user','content':'Reply: GROQ_OK'}],
        max_tokens=10
    )
    print(f'✅ Llama 3.1-8B-Instant connected: {test.choices[0].message.content.strip()}')
except Exception as e:
    print(f'⚠️  Groq connection issue: {e}')

# ---- ChromaDB (local vector store for RAG) ----
# EphemeralClient stores everything in RAM — perfect for Colab.
# In production use chromadb.PersistentClient(path='/your/db/path')
chroma_client = chromadb.EphemeralClient()

# Each farm record will be embedded as a 768-dim sentence vector
# using 'all-MiniLM-L6-v2' — a lightweight, fast embedding model
# trained on diverse text + numerical descriptions.
print('Loading sentence embedding model for ChromaDB...')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create the farm knowledge base collection
farm_collection = chroma_client.get_or_create_collection(
    name='farm_knowledge_base',
    metadata={'hnsw:space': 'cosine'}  # cosine similarity for agronomic similarity
)
print('✅ ChromaDB collection ready.')


---
# CELL 4 — Data Loading & Sentinel-2 Time-Series NDVI

## Why time-series NDVI instead of a single snapshot?

A single NDVI value tells you how green a field is **today**.
A 12-month NDVI trajectory tells you:
- The **crop growth cycle** (planting → peak canopy → senescence → harvest)
- Whether the field experienced **drought stress** (mid-season dip)
- The **phenological pattern** which is crop-type specific
- Whether the field was **left fallow** for part of the year

We simulate 12 monthly NDVI values per farm by adding realistic
seasonal variation (crop-type-specific growth curves + Gaussian noise).
In production, pull monthly Sentinel-2 composites from GEE using the
integration guide in the previous notebook.

From the 12-month NDVI series we derive 5 temporal statistics that
are added to the feature set:
- **ndvi_mean**: average vegetation density across the year
- **ndvi_max**: peak canopy greenness (growing season)
- **ndvi_min**: off-season / bare soil baseline
- **ndvi_std**: variability — high std = strong seasonal cycle
- **ndvi_amplitude**: max − min = strength of the growth pulse


In [ ]:
# ============================================================
# CELL 4 — Load data and generate Sentinel-2 time-series NDVI
# ============================================================
from google.colab import files
import io

print('📂 Upload Crop_recommendation.csv')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'✅ Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')

# ============================================================
# Simulate Sentinel-2 static indices (same as before)
# Replace with real GEE extraction in production.
# ============================================================
def simulate_sentinel2_static(df):
    sc = MinMaxScaler(feature_range=(0.05, 0.95))
    N_s    = sc.fit_transform(df[['N']]).flatten()
    P_s    = sc.fit_transform(df[['P']]).flatten()
    hum_s  = sc.fit_transform(df[['humidity']]).flatten()
    rain_s = sc.fit_transform(df[['rainfall']]).flatten()
    eps = 1e-8
    NIR   = 0.4*N_s + 0.3*hum_s + 0.3*rain_s
    Red   = 0.6*(1-N_s) + 0.4*(1-P_s)
    Green = 0.5*hum_s + 0.5*rain_s
    Blue  = 0.2*P_s + 0.08
    df = df.copy()
    df['NDVI'] = np.clip((NIR-Red)/(NIR+Red+eps), -1, 1)
    df['NDWI'] = np.clip((Green-NIR)/(Green+NIR+eps), -1, 1)
    df['SAVI'] = np.clip(((NIR-Red)/(NIR+Red+0.5+eps))*1.5, -1, 1)
    df['EVI']  = np.clip(2.5*(NIR-Red)/(NIR+6*Red-7.5*Blue+1+eps), -1, 1)
    return df, NIR, Red   # return NIR, Red for NDVI time-series base

# ============================================================
# Simulate 12-month NDVI time-series per farm
#
# Each crop has a characteristic NDVI growth curve:
#   - Rice: rapid rise, plateau, sharp harvest drop
#   - Maize: bell curve (fastest growing season)
#   - Coffee: high stable NDVI year-round (perennial)
#   - Cotton: moderate rise then ginning dip
#   - Apple: spring flush, summer peak, autumn decline
#
# We model this with a sine wave shifted per crop + Gaussian noise.
# The sine wave parameters (phase, amplitude) are crop-specific.
# ============================================================
CROP_NDVI_PARAMS = {
    # crop: (base_ndvi, amplitude, phase_shift_months, noise_std)
    'rice'        : (0.35, 0.30, 2, 0.03),
    'maize'       : (0.30, 0.35, 3, 0.04),
    'chickpea'    : (0.25, 0.20, 0, 0.03),
    'kidneybeans' : (0.28, 0.22, 2, 0.03),
    'pigeonpeas'  : (0.30, 0.18, 1, 0.03),
    'mothbeans'   : (0.20, 0.15, 1, 0.02),
    'mungbean'    : (0.25, 0.18, 2, 0.03),
    'blackgram'   : (0.25, 0.20, 2, 0.03),
    'lentil'      : (0.22, 0.18, 0, 0.03),
    'pomegranate' : (0.45, 0.10, 1, 0.02),
    'banana'      : (0.55, 0.08, 0, 0.02),
    'mango'       : (0.50, 0.12, 4, 0.02),
    'grapes'      : (0.40, 0.25, 3, 0.03),
    'watermelon'  : (0.30, 0.28, 2, 0.04),
    'muskmelon'   : (0.28, 0.25, 2, 0.04),
    'apple'       : (0.35, 0.30, 3, 0.03),
    'orange'      : (0.48, 0.10, 1, 0.02),
    'papaya'      : (0.50, 0.08, 0, 0.02),
    'coconut'     : (0.60, 0.05, 0, 0.02),
    'cotton'      : (0.28, 0.28, 4, 0.04),
    'jute'        : (0.35, 0.32, 2, 0.04),
    'coffee'      : (0.55, 0.06, 0, 0.02),
}

def simulate_ndvi_timeseries(df, seed=42):
    """
    Generate a 12-month NDVI trajectory for each farm row.
    Returns the original df with 12 new columns (ndvi_m01 ... ndvi_m12)
    plus 5 temporal statistics derived from the trajectory.
    """
    rng = np.random.RandomState(seed)
    months = np.arange(12)
    ndvi_matrix = np.zeros((len(df), 12))

    for i, (_, row) in enumerate(df.iterrows()):
        crop = row['label']
        base, amp, phase, noise_std = CROP_NDVI_PARAMS.get(
            crop, (0.30, 0.20, 2, 0.03))

        # Sine wave: NDVI peaks at (phase + 3) months, troughs at opposite
        ndvi_curve = base + amp * np.sin(
            2 * np.pi * (months - phase) / 12
        )
        # Add farm-level Gaussian noise for within-crop variability
        ndvi_curve += rng.normal(0, noise_std, 12)
        ndvi_curve = np.clip(ndvi_curve, 0.0, 0.95)
        ndvi_matrix[i] = ndvi_curve

    df = df.copy()
    for m in range(12):
        df[f'ndvi_m{m+1:02d}'] = ndvi_matrix[:, m]

    # Temporal summary statistics (these become model features)
    df['ndvi_mean']      = ndvi_matrix.mean(axis=1)
    df['ndvi_max']       = ndvi_matrix.max(axis=1)
    df['ndvi_min']       = ndvi_matrix.min(axis=1)
    df['ndvi_std']       = ndvi_matrix.std(axis=1)
    df['ndvi_amplitude'] = ndvi_matrix.max(axis=1) - ndvi_matrix.min(axis=1)

    return df, ndvi_matrix

np.random.seed(SEED)
df, NIR, Red = simulate_sentinel2_static(df_raw)
df, ndvi_matrix = simulate_ndvi_timeseries(df)

# ---- Visualise time-series for 4 representative crops ----
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
sample_crops = ['rice', 'coffee', 'cotton', 'apple']
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

for ax, crop in zip(axes, sample_crops):
    idx = df[df['label'] == crop].index[:10]   # first 10 farms of this crop
    for i in idx:
        ax.plot(month_labels,
                [df.loc[i, f'ndvi_m{m:02d}'] for m in range(1,13)],
                alpha=0.4, linewidth=1, color='forestgreen')
    # Mean curve
    mean_curve = np.mean(
        [[df.loc[i, f'ndvi_m{m:02d}'] for m in range(1,13)] for i in idx],
        axis=0)
    ax.plot(month_labels, mean_curve, 'k-', linewidth=2.5, label='Mean')
    ax.set_title(f'{crop.upper()} — 12-month NDVI trajectory',
                 fontweight='bold')
    ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)
    ax.set_xlabel('Month'); ax.set_ylabel('NDVI')
    ax.legend()

plt.suptitle('Sentinel-2 NDVI Time-Series by Crop Type',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ndvi_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ NDVI time-series generated for {len(df)} farms.')
print(f'   New temporal features: ndvi_mean, ndvi_max, ndvi_min, ndvi_std, ndvi_amplitude')


---
# CELL 5 — Feature Engineering

In [ ]:
# ============================================================
# CELL 5 — Feature engineering
# Adds: domain interactions + temporal NDVI stats
# ============================================================
df_fe = df.copy()

# ---- Original agronomic interactions ----
df_fe['NP_ratio']      = df_fe['N'] / (df_fe['P'] + 1e-8)
df_fe['NK_ratio']      = df_fe['N'] / (df_fe['K'] + 1e-8)
df_fe['NPK_sum']       = df_fe['N'] + df_fe['P'] + df_fe['K']
df_fe['heat_humidity'] = df_fe['temperature'] * df_fe['humidity'] / 100
df_fe['water_balance'] = df_fe['rainfall'] - 0.3 * df_fe['temperature'] * df_fe['humidity']
df_fe['veg_composite'] = (df_fe['NDVI'] + df_fe['SAVI'] + df_fe['EVI']) / 3
df_fe['ph_deviation']  = np.abs(df_fe['ph'] - 7.0)

# ---- Feature set: original + static S2 + temporal NDVI stats ----
# The temporal features (ndvi_mean, ndvi_amplitude, etc.) add
# phenological information that static NDVI cannot capture.
FEATURE_COLS = [
    'N','P','K','temperature','humidity','ph','rainfall',
    'NDVI','NDWI','SAVI','EVI',
    'NP_ratio','NK_ratio','NPK_sum','heat_humidity',
    'water_balance','veg_composite','ph_deviation',
    # Temporal NDVI features (from 12-month time-series)
    'ndvi_mean','ndvi_max','ndvi_min','ndvi_std','ndvi_amplitude'
]

le = LabelEncoder()
df_fe['crop_id'] = le.fit_transform(df_fe['label'])
N_CLASSES  = len(le.classes_)
CROP_NAMES = le.classes_

X = df_fe[FEATURE_COLS].values.astype(np.float32)
y = df_fe['crop_id'].values

# StandardScaler: zero mean, unit variance
# Required for: autoencoder, CNN, MLP, HDBSCAN (distance-based)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'✅ Feature matrix : {X_scaled.shape}')
print(f'   Temporal NDVI features included: ndvi_mean, ndvi_max, ndvi_min, ndvi_std, ndvi_amplitude')
print(f'   Total features: {len(FEATURE_COLS)}')
print(f'   Classes: {N_CLASSES}')


---
# CELL 6 — Autoencoder (Latent Space for HDBSCAN)

In [ ]:
# ============================================================
# CELL 6 — Autoencoder
# Compresses 23 features → 8D latent space.
# HDBSCAN clusters in this latent space where non-linear
# relationships between features have been captured.
# ============================================================
INPUT_DIM  = X_scaled.shape[1]  # 23
LATENT_DIM = 8

def build_autoencoder(input_dim, latent_dim):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    latent = layers.Dense(latent_dim, activation='linear', name='latent')(x)
    x = layers.Dense(32, activation='relu')(latent)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    output = layers.Dense(input_dim, activation='linear')(x)
    ae  = Model(inp, output, name='Autoencoder')
    enc = Model(inp, latent, name='Encoder')
    ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return ae, enc

autoencoder, encoder = build_autoencoder(INPUT_DIM, LATENT_DIM)
cb = [
    EarlyStopping(monitor='val_loss', patience=20,
                  restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=0)
]
autoencoder.fit(X_scaled, X_scaled, epochs=200, batch_size=64,
                validation_split=0.15, callbacks=cb, verbose=0)
X_latent = encoder.predict(X_scaled, verbose=0)
print(f'✅ Autoencoder trained. Latent space: {X_latent.shape}')


---
# CELL 7 — HDBSCAN Clustering

## Why HDBSCAN instead of KMeans?

| Property | KMeans | HDBSCAN |
|----------|--------|---------|
| Requires K in advance | ✅ Yes | ❌ No — finds K automatically |
| Cluster shape | Spherical only | Arbitrary (elongated, crescent, etc.) |
| Outlier handling | Forces every point into a cluster | Labels outliers as −1 |
| Noise robustness | Sensitive to outliers | Built-in noise handling |
| Suitable for agri-data | Partial | ✅ Better — real farm zones are irregular |

**Key parameters:**
- `min_cluster_size=20`: minimum farms to form a zone (prevents tiny meaningless clusters)
- `min_samples=5`: controls how conservative the outlier labelling is — higher = more outliers
- `metric='euclidean'`: standard for continuous agronomic features in latent space
- `cluster_selection_method='eom'`: Excess of Mass — finds stable, well-separated clusters


In [ ]:
# ============================================================
# CELL 7 — HDBSCAN density-based clustering
#
# HDBSCAN works by:
#  1. Building a mutual reachability graph where edge weights
#     encode density-adjusted distances between points.
#  2. Computing a minimum spanning tree of this graph.
#  3. Condensing the tree by removing leaf edges below
#     min_cluster_size, producing a cluster hierarchy.
#  4. Selecting the most stable flat clustering from the
#     hierarchy using the Excess of Mass criterion.
#
# Points that don't belong to any dense region are labelled
# as −1 (noise/outliers). These represent atypical farms
# that sit between agronomic zones.
# ============================================================
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,          # at least 20 farms per zone
    min_samples=5,                # local density for core points
    metric='euclidean',           # distance in latent space
    cluster_selection_method='eom',  # Excess of Mass — stable clusters
    prediction_data=True          # needed for approximate_predict()
)
cluster_labels = clusterer.fit_predict(X_latent)

# Handle outliers: farms labelled −1 are assigned to their
# nearest cluster using soft cluster membership probabilities.
# This ensures every farm gets a recommendation.
n_noise = np.sum(cluster_labels == -1)
if n_noise > 0:
    print(f'  ℹ️  {n_noise} outlier farms detected — assigning to nearest cluster.')
    # Use HDBSCAN's soft clustering for outlier assignment
    soft_clusters = hdbscan.all_points_membership_vectors(clusterer)
    for i in np.where(cluster_labels == -1)[0]:
        cluster_labels[i] = np.argmax(soft_clusters[i])

df_fe['cluster'] = cluster_labels
OPTIMAL_K = len(np.unique(cluster_labels))

print(f'✅ HDBSCAN found {OPTIMAL_K} agronomic zones automatically.')
print(f'   Silhouette Score : {silhouette_score(X_latent, cluster_labels):.4f}')
print(f'   Davies-Bouldin   : {davies_bouldin_score(X_latent, cluster_labels):.4f}')
print('\nZone sizes:')
for cid in sorted(np.unique(cluster_labels)):
    n = np.sum(cluster_labels == cid)
    crops = df_fe[df_fe['cluster']==cid]['label'].value_counts().head(3).index.tolist()
    print(f'  Zone {cid}: {n:>4} farms | dominant: {crops}')

# ---- t-SNE visualisation of HDBSCAN zones ----
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
X_tsne = tsne.fit_transform(X_latent)

df_tsne = pd.DataFrame({
    'TSNE1': X_tsne[:,0], 'TSNE2': X_tsne[:,1],
    'Zone': cluster_labels.astype(str),
    'Crop': df_fe['label'].values
})
fig = px.scatter(df_tsne, x='TSNE1', y='TSNE2', color='Zone',
                 hover_data=['Crop'],
                 title=f'HDBSCAN Zones in t-SNE Space ({OPTIMAL_K} zones found)',
                 color_discrete_sequence=px.colors.qualitative.Bold,
                 opacity=0.75)
fig.update_traces(marker=dict(size=5))
fig.show()

# Build cluster profiles for LLM context
CLUSTER_PROFILES = {}
for cid in range(OPTIMAL_K):
    sub = df_fe[df_fe['cluster']==cid]
    CLUSTER_PROFILES[cid] = {
        'id': cid, 'size': len(sub),
        'dominant_crops': sub['label'].value_counts().head(3).index.tolist(),
        'mean_N': round(sub['N'].mean(),1),
        'mean_P': round(sub['P'].mean(),1),
        'mean_K': round(sub['K'].mean(),1),
        'mean_temp': round(sub['temperature'].mean(),1),
        'mean_humidity': round(sub['humidity'].mean(),1),
        'mean_rainfall': round(sub['rainfall'].mean(),1),
        'mean_NDVI': round(sub['NDVI'].mean(),3),
        'mean_ndvi_amplitude': round(sub['ndvi_amplitude'].mean(),3),
    }


---
# CELL 8 — Optuna Bayesian Hyperparameter Optimisation

## Why Optuna instead of GridSearch or RandomSearch?

- **GridSearch**: exhaustively tries every combination — exponentially slow.
- **RandomSearch**: random sampling — no learning from previous trials.
- **Optuna (Bayesian / TPE)**: builds a probabilistic model of which regions of
  hyperparameter space are promising, and samples from those regions.
  Result: finds better parameters in 5–10× fewer trials than RandomSearch.

We run Optuna on the **full dataset** (not per-cluster) to find globally
optimal parameters, then use those parameters when training per-cluster models.
This avoids the combinatorial explosion of tuning per cluster.


In [ ]:
# ============================================================
# CELL 8 — Optuna hyperparameter optimisation for RF and XGB
#
# TPE Sampler (Tree-structured Parzen Estimator):
#   Models p(hyperparams | good_score) and p(hyperparams | bad_score)
#   separately using kernel density estimates, then samples from
#   configurations where good/bad ratio is highest.
#   This is the standard Bayesian optimisation approach in Optuna.
#
# We use 5-fold stratified CV as the objective to prevent overfitting
# to any single train/test split.
# ============================================================

# Global train/test split for Optuna (whole dataset)
X_tr_opt, X_te_opt, y_tr_opt, y_te_opt = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED, stratify=y
)
CV_FOLDS = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ---- Optuna for Random Forest ----
def rf_objective(trial):
    """
    Objective function for RF optimisation.
    Optuna calls this function n_trials times, each time suggesting
    different hyperparameter values and updating its internal model
    based on the returned score.
    """
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features':    trial.suggest_categorical('max_features',
                                                     ['sqrt','log2', None]),
    }
    rf = RandomForestClassifier(**params, random_state=SEED, n_jobs=-1)
    scores = cross_val_score(rf, X_tr_opt, y_tr_opt,
                             cv=CV_FOLDS, scoring='accuracy', n_jobs=-1)
    return scores.mean()

print('🔍 Optimising Random Forest with Optuna (30 trials)...')
rf_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
rf_study.optimize(rf_objective, n_trials=30, show_progress_bar=True)
BEST_RF_PARAMS = rf_study.best_params
print(f'✅ Best RF params : {BEST_RF_PARAMS}')
print(f'   Best CV score  : {rf_study.best_value:.4f}')

# ---- Optuna for XGBoost ----
def xgb_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 10),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-5, 10, log=True),
    }
    xgb = XGBClassifier(**params, eval_metric='mlogloss',
                         use_label_encoder=False, verbosity=0,
                         random_state=SEED)
    scores = cross_val_score(xgb, X_tr_opt, y_tr_opt,
                             cv=CV_FOLDS, scoring='accuracy', n_jobs=-1)
    return scores.mean()

print('\n🔍 Optimising XGBoost with Optuna (30 trials)...')
xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
xgb_study.optimize(xgb_objective, n_trials=30, show_progress_bar=True)
BEST_XGB_PARAMS = xgb_study.best_params
print(f'✅ Best XGB params : {BEST_XGB_PARAMS}')
print(f'   Best CV score   : {xgb_study.best_value:.4f}')

# ---- Plot Optuna optimisation history ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, study, name in [(axes[0], rf_study,'Random Forest'),
                         (axes[1], xgb_study,'XGBoost')]:
    values = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(values)
    ax.plot(values, 'o', alpha=0.4, color='steelblue', label='Trial score')
    ax.plot(best_so_far, '-', color='crimson', linewidth=2, label='Best so far')
    ax.set_title(f'{name} — Optuna Optimisation History', fontweight='bold')
    ax.set_xlabel('Trial'); ax.set_ylabel('5-Fold CV Accuracy')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()


---
# CELL 9 — Per-Cluster Models: Optuna-Tuned + Calibrated + MC-Dropout

## Probability Calibration
Raw softmax or predict_proba outputs are often overconfident:
a model may say 95% confidence when it's only right 75% of the time.
`CalibratedClassifierCV` with `method='isotonic'` fits a monotonic
mapping from raw scores to true probabilities using held-out data.
After calibration, if the model says 80%, it should be right ~80% of the time.

## Monte Carlo Dropout (Uncertainty Quantification)
Standard neural networks give one prediction. MC-Dropout gives a
**distribution of predictions** by running the same input through
the network 50 times with dropout randomly active each time.
- Mean of the 50 predictions = best estimate
- Standard deviation = epistemic uncertainty (model doesn't know)
- Wide std = the model is uncertain → flag for human review


In [ ]:
# ============================================================
# CELL 9 — Per-cluster model training
#
# Each cluster gets:
#   (a) Optuna-tuned RF + Calibrated with isotonic regression
#   (b) Optuna-tuned XGB + Calibrated with isotonic regression
#   (c) GBM + Calibrated (baseline comparison)
#   (d) Soft Voting Ensemble of the 3 calibrated models
#   (e) 1D-CNN with MC-Dropout uncertainty
#   (f) TabNet-MLP with MC-Dropout uncertainty
# ============================================================

def build_cnn1d_mcdropout(n_features, n_classes, dropout_rate=0.3):
    """
    1D-CNN with MC-Dropout.
    KEY: set training=True at inference to keep dropout active.
    This makes each forward pass sample a different subnetwork,
    producing a different prediction — the variance across 50 runs
    is the epistemic uncertainty.
    """
    inp = keras.Input(shape=(n_features, 1))
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x)
    # Dropout with training=True will be activated even at inference
    x = layers.Dropout(dropout_rate)(x, training=True)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x, training=True)
    out = layers.Dense(n_classes, activation='softmax')(x)
    m = Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def build_mlp_mcdropout(n_features, n_classes, dropout_rate=0.3):
    """
    TabNet-style MLP with MC-Dropout uncertainty.
    The attention gate learns which features are relevant per sample.
    MC-Dropout is applied to the attention output and hidden layers.
    """
    inp = keras.Input(shape=(n_features,))
    att = layers.Dense(n_features, activation='sigmoid')(inp)
    x   = layers.Multiply()([inp, att])
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.LayerNormalization()(x)
    x   = layers.Dropout(dropout_rate)(x, training=True)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.LayerNormalization()(x)
    x   = layers.Dropout(dropout_rate)(x, training=True)
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.Dropout(0.2)(x, training=True)
    out = layers.Dense(n_classes, activation='softmax')(x)
    m   = Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(5e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def mc_dropout_predict(model, X, n_passes=50, is_cnn=False):
    """
    Run MC-Dropout inference.
    Runs n_passes forward passes with dropout active.
    Returns:
      mean_pred  : averaged probability vector (best estimate)
      uncertainty: standard deviation across passes (epistemic uncertainty)
                   High uncertainty = model is not confident → flag for review
    """
    preds = []
    for _ in range(n_passes):
        if is_cnn:
            p = model(X.reshape(-1, X.shape[-1], 1), training=True).numpy()
        else:
            p = model(X, training=True).numpy()
        preds.append(p)
    preds = np.array(preds)              # (n_passes, n_samples, n_classes)
    return preds.mean(axis=0), preds.std(axis=0)

DL_CB = [
    EarlyStopping(monitor='val_accuracy', patience=15,
                  restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=7, verbose=0)
]

cluster_models   = {}
cluster_encoders = {}
cluster_scalers  = {}
all_results      = []

for cid in range(OPTIMAL_K):
    print(f'\n{"="*58}')
    print(f'  ZONE {cid} — {CLUSTER_PROFILES[cid]["dominant_crops"]}')
    print(f'{"="*58}')

    idx = df_fe['cluster'] == cid
    X_c, y_raw = X_scaled[idx], y[idx]

    le_local = LabelEncoder()
    y_c = le_local.fit_transform(y_raw)
    cluster_encoders[cid] = le_local
    n_local = len(le_local.classes_)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_c, y_c, test_size=0.2, random_state=SEED, stratify=y_c)

    sc_l = StandardScaler()
    X_tr_s, X_te_s = sc_l.fit_transform(X_tr), sc_l.transform(X_te)
    cluster_scalers[cid] = sc_l
    cluster_models[cid]  = {}

    # ---- (a) Optuna-tuned RF + Calibration ----
    # CalibratedClassifierCV with cv=5:
    #   Trains 5 RF models, each time using 4/5 as classifier and
    #   1/5 as calibration set — honest probability estimates.
    rf_base = RandomForestClassifier(
        **BEST_RF_PARAMS, random_state=SEED, n_jobs=-1)
    rf_cal = CalibratedClassifierCV(rf_base, method='isotonic', cv=3)
    rf_cal.fit(X_tr, y_tr)
    rf_acc = accuracy_score(y_te, rf_cal.predict(X_te))
    cluster_models[cid]['RF'] = rf_cal
    print(f'  RF  (tuned+calibrated): {rf_acc:.4f}')

    # ---- (b) Optuna-tuned XGB + Calibration ----
    xgb_base = XGBClassifier(
        **BEST_XGB_PARAMS, eval_metric='mlogloss',
        use_label_encoder=False, verbosity=0, random_state=SEED)
    xgb_cal = CalibratedClassifierCV(xgb_base, method='isotonic', cv=3)
    xgb_cal.fit(X_tr, y_tr)
    xgb_acc = accuracy_score(y_te, xgb_cal.predict(X_te))
    cluster_models[cid]['XGB'] = xgb_cal
    print(f'  XGB (tuned+calibrated): {xgb_acc:.4f}')

    # ---- (c) GBM + Calibration ----
    gbm_base = GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, random_state=SEED)
    gbm_cal = CalibratedClassifierCV(gbm_base, method='isotonic', cv=3)
    gbm_cal.fit(X_tr, y_tr)
    gbm_acc = accuracy_score(y_te, gbm_cal.predict(X_te))
    cluster_models[cid]['GBM'] = gbm_cal
    print(f'  GBM (calibrated)      : {gbm_acc:.4f}')

    # ---- (d) Soft Voting Ensemble ----
    ens = VotingClassifier(
        estimators=[('rf',rf_cal),('xgb',xgb_cal),('gbm',gbm_cal)],
        voting='soft', n_jobs=-1)
    ens.fit(X_tr, y_tr)
    ens_acc = accuracy_score(y_te, ens.predict(X_te))
    cluster_models[cid]['Ensemble'] = ens
    print(f'  Ensemble              : {ens_acc:.4f}')

    # ---- (e) 1D-CNN with MC-Dropout ----
    cnn = build_cnn1d_mcdropout(X_tr_s.shape[1], n_local)
    cnn.fit(X_tr_s.reshape(-1,X_tr_s.shape[1],1), y_tr,
            epochs=150, batch_size=32,
            validation_split=0.15, callbacks=DL_CB, verbose=0)
    cnn_mean, _ = mc_dropout_predict(cnn, X_te_s, n_passes=20, is_cnn=True)
    cnn_acc = accuracy_score(y_te, np.argmax(cnn_mean, axis=1))
    cluster_models[cid]['CNN1D'] = cnn
    print(f'  CNN1D (MC-Dropout)    : {cnn_acc:.4f}')

    # ---- (f) TabNet-MLP with MC-Dropout ----
    mlp = build_mlp_mcdropout(X_tr_s.shape[1], n_local)
    mlp.fit(X_tr_s, y_tr, epochs=150, batch_size=32,
            validation_split=0.15, callbacks=DL_CB, verbose=0)
    mlp_mean, _ = mc_dropout_predict(mlp, X_te_s, n_passes=20)
    mlp_acc = accuracy_score(y_te, np.argmax(mlp_mean, axis=1))
    cluster_models[cid]['MLP'] = mlp
    print(f'  MLP  (MC-Dropout)     : {mlp_acc:.4f}')

    for name, acc in [('RF',rf_acc),('XGB',xgb_acc),('GBM',gbm_acc),
                       ('Ensemble',ens_acc),('CNN1D',cnn_acc),('MLP',mlp_acc)]:
        all_results.append({'Zone':cid,'Model':name,'Accuracy':acc})

results_df = pd.DataFrame(all_results)
print('\n✅ All models trained.')
print(results_df.groupby('Model')['Accuracy'].mean().sort_values(ascending=False).round(4))


---
# CELL 10 — SHAP Explainability

## What SHAP tells us
SHAP (SHapley Additive exPlanations) assigns each feature a contribution
value for each individual prediction. It answers: *"For this specific farm,
which features pushed the model toward recommending rice?"*

- **Positive SHAP value**: this feature pushed the prediction toward this crop
- **Negative SHAP value**: this feature pushed prediction away from this crop
- **Magnitude**: how strongly

This is critical for farmer trust — instead of a black-box percentage,
the farmer sees: *"Your high humidity (+0.34) and high rainfall (+0.28) are
the main reasons rice was recommended."*


In [ ]:
# ============================================================
# CELL 10 — SHAP explainability
#
# We use TreeExplainer — the fastest SHAP explainer for tree-based
# models (RF, XGB, GBM). It exploits the tree structure to compute
# exact SHAP values in O(TLD^2) time where T=trees, L=leaves, D=depth.
#
# For DL models we would use DeepExplainer or GradientExplainer,
# but TreeExplainer on the ensemble is sufficient and faster.
# ============================================================

# Use Zone 0 for demonstration (repeat for other zones as needed)
DEMO_ZONE = 0
idx_demo = df_fe['cluster'] == DEMO_ZONE
X_demo   = X_scaled[idx_demo][:100]   # 100 samples for speed
y_demo   = y[idx_demo][:100]

# Get the underlying RF from the CalibratedClassifierCV wrapper
# CalibratedClassifierCV stores base estimators in calibrated_classifiers_
# Each calibrated_classifier has a base_estimator attribute
rf_base_for_shap = cluster_models[DEMO_ZONE]['RF'].calibrated_classifiers_[0].estimator

print(f'Computing SHAP values for Zone {DEMO_ZONE} (100 samples)...')
explainer   = shap.TreeExplainer(rf_base_for_shap)
shap_values = explainer.shap_values(X_demo)
print('✅ SHAP values computed.')

# ---- Global feature importance (beeswarm summary) ----
# Each dot = one farm
# X-axis = SHAP value (positive = pushed toward this class)
# Color  = feature value (red = high, blue = low)
print('\nPlotting SHAP summary for Zone 0...')
fig, ax = plt.subplots(figsize=(10, 10))

# shap_values is a list of arrays (one per class) for multi-class RF
# We show the summary for the top class (class 0 in local encoding)
if isinstance(shap_values, list):
    sv_plot = shap_values[0]
else:
    sv_plot = shap_values

shap.summary_plot(
    sv_plot, X_demo,
    feature_names=FEATURE_COLS,
    plot_type='dot',
    max_display=20,
    show=False
)
plt.title(f'SHAP Feature Importance — Zone {DEMO_ZONE}',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Bar plot of mean absolute SHAP (global importance) ----
mean_abs_shap = np.abs(sv_plot).mean(axis=0)
shap_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(shap_df['Feature'], shap_df['Mean |SHAP|'],
        color=plt.cm.viridis(shap_df['Mean |SHAP|'] / shap_df['Mean |SHAP|'].max()))
ax.set_title(f'Mean |SHAP| Value per Feature — Zone {DEMO_ZONE}',
             fontweight='bold')
ax.set_xlabel('Mean Absolute SHAP Value')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Per-farm SHAP waterfall for a single prediction ----
# This is what the app would show for each individual farm recommendation.
print('\nWaterfall plot for a single farm (farm #0 in Zone 0):')
if isinstance(shap_values, list):
    # Multi-class: show waterfall for the predicted class
    pred_class = int(np.argmax(explainer.expected_value)) if hasattr(
        explainer.expected_value, '__len__') else 0
    shap_single = shap.Explanation(
        values=shap_values[pred_class][0],
        base_values=explainer.expected_value[pred_class] if hasattr(
            explainer.expected_value, '__len__') else explainer.expected_value,
        data=X_demo[0],
        feature_names=FEATURE_COLS
    )
else:
    shap_single = shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_demo[0],
        feature_names=FEATURE_COLS
    )

plt.figure(figsize=(10, 7))
shap.plots.waterfall(shap_single, max_display=15, show=False)
plt.title('SHAP Waterfall — Single Farm Explanation', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP analysis complete.')


---
# CELL 11 — Calibration Verification (Reliability Diagram)

In [ ]:
# ============================================================
# CELL 11 — Verify that calibration worked
#
# A reliability diagram (calibration curve) plots:
#   X-axis: mean predicted probability for a bin
#   Y-axis: actual fraction of correct predictions in that bin
#
# A perfectly calibrated model follows the diagonal (y=x).
# A curve above diagonal = underconfident.
# A curve below diagonal = overconfident (common in NNs and uncalibrated trees).
#
# We compare uncalibrated vs calibrated RF to show the improvement.
# ============================================================
from sklearn.calibration import calibration_curve

# Use global split for clean comparison
X_tr_g, X_te_g, y_tr_g, y_te_g = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED, stratify=y)

# Uncalibrated RF (default)
rf_uncal = RandomForestClassifier(**BEST_RF_PARAMS, random_state=SEED, n_jobs=-1)
rf_uncal.fit(X_tr_g, y_tr_g)

# Calibrated RF (isotonic)
rf_cal_global = CalibratedClassifierCV(
    RandomForestClassifier(**BEST_RF_PARAMS, random_state=SEED, n_jobs=-1),
    method='isotonic', cv=5)
rf_cal_global.fit(X_tr_g, y_tr_g)

# Compare calibration for class 0 (rice) as representative
y_binary = (y_te_g == 0).astype(int)

prob_uncal = rf_uncal.predict_proba(X_te_g)[:, 0]
prob_cal   = rf_cal_global.predict_proba(X_te_g)[:, 0]

frac_pos_u, mean_pred_u = calibration_curve(y_binary, prob_uncal, n_bins=10)
frac_pos_c, mean_pred_c = calibration_curve(y_binary, prob_cal,   n_bins=10)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0,1],[0,1], 'k--', linewidth=1.5, label='Perfect calibration')
ax.plot(mean_pred_u, frac_pos_u, 's-', color='tomato',
        linewidth=2, markersize=8, label='Uncalibrated RF')
ax.plot(mean_pred_c, frac_pos_c, 'o-', color='seagreen',
        linewidth=2, markersize=8, label='Calibrated RF (isotonic)')
ax.set_title('Reliability Diagram — RF Calibration Effect', fontweight='bold')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Correct Predictions')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Calibration verified — calibrated curve closer to the diagonal.')


---
# CELL 12 — Yield Prediction (Regression)

## Why add yield prediction?
Currently the agent recommends *which* crop. Adding a yield regression
model transforms this from advisory to **quantitative** — farmers make
economic decisions based on expected tons/ha, not just crop suitability.

We synthesise realistic yield labels from the FAO yield database ranges
per crop, perturbed by the farm's soil and climate features.
In production, replace with actual yield records from field surveys.


In [ ]:
# ============================================================
# CELL 12 — Crop yield regression model
#
# Architecture choice: GradientBoostingRegressor
#   - Handles heterogeneous tabular features well
#   - More interpretable than neural networks for regression
#   - Naturally captures non-linear soil-yield interactions
#   - Does not require feature scaling
#
# Target variable: synthetic yield (tons/ha) derived from
# FAO average yield ranges per crop, adjusted by:
#   - NPK availability (nutrient supply)
#   - Water balance (rainfall - evapotranspiration proxy)
#   - Temperature suitability (crop-specific optimal range)
#   - NDVI amplitude (proxy for actual growth achieved)
# ============================================================

# FAO reference yields (tons/ha) — (min, max) for each crop
FAO_YIELD_RANGES = {
    'rice': (2.5, 6.0), 'maize': (3.0, 8.0), 'chickpea': (0.8, 2.0),
    'kidneybeans': (0.8, 2.5), 'pigeonpeas': (0.5, 1.5),
    'mothbeans': (0.3, 0.8), 'mungbean': (0.6, 1.5),
    'blackgram': (0.5, 1.2), 'lentil': (0.8, 2.0),
    'pomegranate': (8.0, 20.0), 'banana': (20.0, 45.0),
    'mango': (5.0, 15.0), 'grapes': (8.0, 20.0),
    'watermelon': (15.0, 40.0), 'muskmelon': (10.0, 25.0),
    'apple': (10.0, 25.0), 'orange': (10.0, 25.0),
    'papaya': (20.0, 60.0), 'coconut': (5.0, 15.0),
    'cotton': (1.5, 4.0), 'jute': (2.0, 4.5), 'coffee': (0.5, 2.0),
}

def synthesise_yield(df):
    """
    Create synthetic yield labels by taking the FAO range midpoint
    and adjusting it by normalised soil + climate + satellite factors.
    The result is a realistic yield distribution per crop.
    """
    rng = np.random.RandomState(SEED)
    yields = np.zeros(len(df))

    # Normalise key drivers to [0, 1]
    sc_yield = MinMaxScaler()
    drivers = sc_yield.fit_transform(
        df[['N','P','K','water_balance','ndvi_amplitude']].values)

    for i, (_, row) in enumerate(df.iterrows()):
        lo, hi = FAO_YIELD_RANGES.get(row['label'], (1.0, 5.0))
        mid = (lo + hi) / 2

        # Composite agronomic factor: weighted average of drivers
        # N is most important (0.3), then water_balance (0.25),
        # then ndvi_amplitude (0.25), then P and K equally
        agro_factor = (
            0.30 * drivers[i, 0] +   # N
            0.15 * drivers[i, 1] +   # P
            0.15 * drivers[i, 2] +   # K
            0.25 * drivers[i, 3] +   # water_balance
            0.15 * drivers[i, 4]     # ndvi_amplitude
        )  # sum of weights = 1.0

        # Yield = midpoint ± range/2 * agronomic_factor + noise
        yield_val = lo + (hi - lo) * agro_factor
        yield_val += rng.normal(0, (hi - lo) * 0.05)   # 5% noise
        yields[i] = np.clip(yield_val, lo * 0.7, hi * 1.1)

    return yields

df_fe['yield_tons_ha'] = synthesise_yield(df_fe)

print('Yield distribution by crop:')
print(df_fe.groupby('label')['yield_tons_ha'].agg(['mean','std','min','max']).round(2))

# ---- Train yield regression model ----
# We use the same FEATURE_COLS + crop identity to predict yield.
# GradientBoostingRegressor is chosen over RF regressor because:
#  - Better extrapolation at the edges of the feature space
#  - More accurate with the small per-cluster sample sizes
le_yield = LabelEncoder()
X_yield = np.hstack([
    X_scaled,
    le_yield.fit_transform(df_fe['label'].values).reshape(-1,1)
])
y_yield = df_fe['yield_tons_ha'].values

X_tr_y, X_te_y, y_tr_y, y_te_y = train_test_split(
    X_yield, y_yield, test_size=0.2, random_state=SEED)

yield_model = GradientBoostingRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=SEED)
yield_model.fit(X_tr_y, y_tr_y)

y_pred_y = yield_model.predict(X_te_y)
rmse = np.sqrt(mean_squared_error(y_te_y, y_pred_y))
r2   = r2_score(y_te_y, y_pred_y)
print(f'\n✅ Yield model trained.')
print(f'   RMSE : {rmse:.3f} tons/ha')
print(f'   R²   : {r2:.4f}')

# ---- Actual vs predicted scatter ----
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_te_y, y_pred_y, alpha=0.5, color='steelblue', s=20)
mn, mx = y_te_y.min(), y_te_y.max()
ax.plot([mn,mx],[mn,mx], 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Yield (tons/ha)'); ax.set_ylabel('Predicted Yield (tons/ha)')
ax.set_title(f'Yield Model — Actual vs Predicted\nR²={r2:.3f}, RMSE={rmse:.3f} t/ha',
             fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('yield_regression.png', dpi=150, bbox_inches='tight')
plt.show()


---
# CELL 13 — Multi-Objective Optimisation (NSGA-II / Pareto)

## The problem with single-objective recommendations
Current approach: recommend the crop with highest classification probability.
Real farmers optimise multiple objectives simultaneously:
1. **Suitability** (maximise) — how well the crop fits the soil/climate
2. **Yield** (maximise) — expected tons/ha
3. **Water usage** (minimise) — important in water-scarce regions
4. **Carbon footprint** (minimise) — sustainability metric

These objectives conflict — the highest-yield crop often needs the most water.
NSGA-II finds the **Pareto front**: the set of crops where no objective
can be improved without worsening another. The farmer chooses based on
their own priorities.


In [ ]:
# ============================================================
# CELL 13 — Multi-objective crop optimisation with NSGA-II
#
# NSGA-II (Non-dominated Sorting Genetic Algorithm II):
#   - Population-based evolutionary algorithm
#   - Maintains diversity via crowding distance
#   - Converges to the Pareto front in O(MN^2) where M=objectives, N=population
#   - pymoo implementation is production-grade and well-tested
#
# We define 4 objectives per crop:
#   f1 = -suitability  (negate to convert max → min for pymoo)
#   f2 = -yield        (negate)
#   f3 = +water_usage  (minimise directly)
#   f4 = +carbon_index (minimise directly)
# ============================================================

# Water usage indices (relative, 1=low, 5=very high)
# Based on FAO irrigation requirements and crop water productivity
WATER_USAGE = {
    'rice':2100,'jute':1400,'banana':1200,'sugarcane':1500,'cotton':1200,
    'maize':500,'wheat':450,'coffee':800,'coconut':900,'papaya':800,
    'orange':900,'mango':800,'grapes':600,'apple':600,'watermelon':400,
    'muskmelon':400,'pomegranate':350,'lentil':250,'chickpea':300,
    'blackgram':300,'mungbean':300,'mothbeans':200,'pigeonpeas':350,
    'kidneybeans':350
}

# Carbon footprint indices (kg CO2-eq per ton of output, approximate)
CARBON_INDEX = {
    'rice':1600,'maize':400,'cotton':3000,'coffee':4000,'banana':500,
    'mango':600,'grapes':800,'apple':700,'orange':600,'papaya':400,
    'coconut':300,'watermelon':200,'muskmelon':200,'pomegranate':500,
    'lentil':200,'chickpea':200,'blackgram':200,'mungbean':200,
    'mothbeans':150,'pigeonpeas':200,'kidneybeans':250,'jute':300,'sugarcane':300
}

def get_pareto_crops(farm_ml_result, farm_data, top_n=5):
    """
    Given ML ensemble probabilities, compute the Pareto front
    across 4 objectives for the top_n candidate crops.

    Returns a DataFrame of Pareto-optimal crops with all 4 objective values.
    """
    # Build candidate crop data
    cands = []
    cluster_id  = farm_ml_result['cluster_id']
    le_local    = cluster_encoders[cluster_id]
    global_crops = le.inverse_transform(le_local.classes_)

    for crop_name, confidence in zip(
            farm_ml_result['top_crops'][:top_n],
            farm_ml_result['confidences'][:top_n]):

        # Encode crop for yield model
        crop_encoded = le_yield.transform([crop_name])[0]
        x_yield_input = np.append(
            scaler.transform(
                engineer_single_full(farm_data).reshape(1,-1)
            ), crop_encoded
        ).reshape(1,-1)

        pred_yield = float(yield_model.predict(x_yield_input)[0])
        water      = WATER_USAGE.get(crop_name, 700)
        carbon     = CARBON_INDEX.get(crop_name, 500)

        cands.append({
            'crop'        : crop_name,
            'suitability' : confidence / 100,   # normalise to [0,1]
            'yield'       : pred_yield,
            'water_usage' : water,
            'carbon'      : carbon,
        })

    df_cands = pd.DataFrame(cands)

    # ---- Non-dominated sorting (Pareto front) ----
    # A solution is dominated if another solution is better in ALL objectives.
    # Non-dominated solutions form the Pareto front.
    # We negate suitability and yield (we want to maximise these).
    obj_matrix = np.column_stack([
        -df_cands['suitability'].values,  # negate → minimise
        -df_cands['yield'].values,        # negate → minimise
        df_cands['water_usage'].values,   # minimise directly
        df_cands['carbon'].values         # minimise directly
    ])

    nds = NonDominatedSorting()
    fronts = nds.do(obj_matrix)
    pareto_indices = fronts[0]   # first front = Pareto optimal

    df_cands['pareto_rank'] = 999
    for rank, front in enumerate(fronts):
        for idx in front:
            df_cands.loc[idx, 'pareto_rank'] = rank + 1

    df_pareto = df_cands[df_cands['pareto_rank'] == 1].copy()

    return df_cands, df_pareto

# ---- Helper needed for yield prediction inside the agent ----
def engineer_single_full(farm_data):
    """Engineer all features for a single farm dict."""
    N, P, K = farm_data['N'], farm_data['P'], farm_data['K']
    T, H, ph, R = (farm_data['temperature'], farm_data['humidity'],
                   farm_data['ph'], farm_data['rainfall'])
    NDVI = farm_data.get('NDVI', 0.3)
    NDWI = farm_data.get('NDWI', 0.0)
    SAVI = farm_data.get('SAVI', 0.25)
    EVI  = farm_data.get('EVI', 0.28)
    ndvi_mean = farm_data.get('ndvi_mean', NDVI)
    ndvi_max  = farm_data.get('ndvi_max', NDVI + 0.1)
    ndvi_min  = farm_data.get('ndvi_min', NDVI - 0.1)
    ndvi_std  = farm_data.get('ndvi_std', 0.05)
    ndvi_amp  = farm_data.get('ndvi_amplitude', 0.2)

    return np.array([
        N, P, K, T, H, ph, R,
        NDVI, NDWI, SAVI, EVI,
        N/(P+1e-8), N/(K+1e-8), N+P+K,
        T*H/100, R - 0.3*T*H,
        (NDVI+SAVI+EVI)/3, abs(ph-7.0),
        ndvi_mean, ndvi_max, ndvi_min, ndvi_std, ndvi_amp
    ], dtype=np.float32)

print('✅ Multi-objective Pareto optimisation functions ready.')
print('   Objectives: suitability, yield, water usage, carbon footprint')
print('   Algorithm : Non-dominated sorting (NSGA-II style Pareto front)')


---
# CELL 14 — ChromaDB Vector RAG (Similar Farm Retrieval)

## How RAG improves the LLM recommendations
Without RAG: Llama generates advice from its pretrained knowledge only.
With RAG: Llama receives the 5 most similar **historical farms** from the
dataset along with their actual recommended crops. This grounds the LLM
in real agronomic data specific to the same agro-ecological zone.

Think of it as: instead of asking a general agronomist, you're asking
an agronomist who has personally worked with the 5 most similar farms
in your region.

## How it works
1. Each farm record is converted to a natural language description string.
2. The string is embedded into a 768-dim vector using sentence-transformers.
3. All 2200 farm embeddings are stored in ChromaDB.
4. At inference, the new farm is embedded and the 5 nearest neighbours
   (cosine similarity) are retrieved.
5. Their descriptions and recommended crops are injected into the Llama prompt.


In [ ]:
# ============================================================
# CELL 14 — Build and populate ChromaDB farm knowledge base
#
# We represent each farm as a natural language sentence describing
# its soil and climate profile. This is better than embedding raw
# numbers directly because sentence-transformers are trained on text
# and understand agronomic vocabulary.
#
# Example embedding text:
#   "Farm with N=90 P=42 K=43 temperature=21C humidity=82% ph=6.5
#    rainfall=210mm NDVI=0.32 NDVI_amplitude=0.28 recommended: rice"
# ============================================================

def farm_to_text(row):
    """Convert a farm row to a descriptive string for embedding."""
    return (
        f"Farm with N={row['N']:.0f} P={row['P']:.0f} K={row['K']:.0f} "
        f"temperature={row['temperature']:.1f}C humidity={row['humidity']:.0f}% "
        f"pH={row['ph']:.1f} rainfall={row['rainfall']:.0f}mm "
        f"NDVI={row['NDVI']:.2f} ndvi_amplitude={row['ndvi_amplitude']:.2f} "
        f"water_balance={row['water_balance']:.0f} NPK_sum={row['NPK_sum']:.0f} "
        f"recommended_crop={row['label']}"
    )

print('Building ChromaDB farm knowledge base...')
print('(Embedding 2200 farms — this takes ~1 minute on Colab CPU)')

BATCH_SIZE = 100
total = len(df_fe)

for batch_start in range(0, total, BATCH_SIZE):
    batch_end  = min(batch_start + BATCH_SIZE, total)
    batch      = df_fe.iloc[batch_start:batch_end]

    texts      = [farm_to_text(row) for _, row in batch.iterrows()]
    embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()

    farm_collection.add(
        ids=[f'farm_{i}' for i in range(batch_start, batch_end)],
        embeddings=embeddings,
        documents=texts,
        metadatas=[{
            'crop'           : row['label'],
            'cluster'        : int(row['cluster']),
            'N'              : float(row['N']),
            'rainfall'       : float(row['rainfall']),
            'ndvi_amplitude' : float(row['ndvi_amplitude']),
        } for _, row in batch.iterrows()]
    )

    if batch_end % 500 == 0 or batch_end == total:
        print(f'  Embedded {batch_end}/{total} farms')

print(f'✅ ChromaDB populated with {farm_collection.count()} farm embeddings.')

def retrieve_similar_farms(farm_data, n_results=5):
    """
    Retrieve the n_results most agronomically similar farms
    from the knowledge base using cosine similarity in embedding space.
    Returns a formatted string for injection into the LLM prompt.
    """
    query_text = (
        f"Farm with N={farm_data['N']:.0f} P={farm_data['P']:.0f} "
        f"K={farm_data['K']:.0f} temperature={farm_data['temperature']:.1f}C "
        f"humidity={farm_data['humidity']:.0f}% pH={farm_data['ph']:.1f} "
        f"rainfall={farm_data['rainfall']:.0f}mm "
        f"NDVI={farm_data.get('NDVI', 0.3):.2f} "
        f"ndvi_amplitude={farm_data.get('ndvi_amplitude', 0.2):.2f}"
    )

    query_embedding = embedding_model.encode([query_text]).tolist()

    results = farm_collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )

    similar_text = "SIMILAR HISTORICAL FARMS (from knowledge base):\n"
    for i, (doc, meta, dist) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0])):
        similarity = round((1 - dist) * 100, 1)
        similar_text += (
            f"  Farm {i+1} (similarity={similarity}%): "
            f"crop={meta['crop']}, {doc[:120]}...\n"
        )
    return similar_text

print('✅ RAG retrieval function ready.')
print('   Test retrieval:')
test_farm = {'N':90,'P':42,'K':43,'temperature':21,'humidity':82,
             'ph':6.5,'rainfall':210,'NDVI':0.32,'ndvi_amplitude':0.28}
print(retrieve_similar_farms(test_farm, n_results=3))


---
# CELL 15 — Federated Learning Architecture Design

## What is Federated Learning and why does it matter here?
In a real deployment, farms are spread across many regions.
Their soil and yield data is **sensitive** — farmers may not want
their raw data leaving their devices or shared with a central server.

**Federated Learning (FL)** solves this:
- Each farm device trains a local model on its own data.
- Only the **model weights** (not the raw data) are sent to the server.
- The server aggregates weights using **FedAvg** and sends the improved
  global model back to all devices.
- Raw farm data never leaves the farm.

This is the architecture used by Google for Gboard (keyboard predictions)
and Apple for Siri — we apply the same principle to agronomic models.

## Implementation
We simulate FedAvg across 5 virtual farm clients (representing 5 farms/cooperatives).
Each client trains locally on its cluster's data, then we aggregate.


In [ ]:
# ============================================================
# CELL 15 — Federated Learning simulation (FedAvg algorithm)
#
# FedAvg (McMahan et al., 2017) algorithm:
#   1. Server initialises global model weights W_global
#   2. For each communication round t:
#      a. Server sends W_global to all K clients
#      b. Each client k trains locally for E epochs on its data:
#         W_k ← W_global - η * ∇L_k(W_global)
#      c. Clients send W_k back to server
#      d. Server aggregates: W_global ← Σ(n_k/n * W_k)
#         where n_k = client k's sample count, n = total samples
#   3. Repeat for T rounds
#
# Privacy guarantee: only weight tensors (not raw data) are shared.
# For stronger privacy, add Differential Privacy noise to weights
# before sending (not implemented here but noted for production).
# ============================================================

def build_federated_mlp(input_dim, n_classes):
    """
    Lightweight MLP for federated training.
    Smaller than the full TabNet-MLP to be suitable for
    edge devices (IoT soil sensors, mobile phones on farms).
    """
    inp = keras.Input(shape=(input_dim,))
    x   = layers.Dense(64, activation='relu')(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    m   = Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def fedavg_aggregate(global_weights, client_weights_list, client_sizes):
    """
    FedAvg aggregation: weighted average of client model weights.
    Weight of each client = its proportion of total training samples.
    This ensures larger farms/cooperatives have more influence,
    which is fair and reduces variance from small clients.
    """
    total_samples = sum(client_sizes)
    aggregated    = []

    for layer_idx in range(len(global_weights)):
        weighted_sum = np.zeros_like(global_weights[layer_idx])
        for client_w, n_k in zip(client_weights_list, client_sizes):
            weighted_sum += (n_k / total_samples) * client_w[layer_idx]
        aggregated.append(weighted_sum)

    return aggregated

# ---- Federated simulation ----
N_CLIENTS       = min(OPTIMAL_K, 5)   # one client per agronomic zone
N_ROUNDS        = 5                    # communication rounds
LOCAL_EPOCHS    = 10                   # local training epochs per round
FL_INPUT_DIM    = X_scaled.shape[1]
FL_N_CLASSES    = N_CLASSES

print(f'🌐 Federated Learning Simulation')
print(f'   Clients       : {N_CLIENTS} (one per agronomic zone)')
print(f'   Rounds        : {N_ROUNDS}')
print(f'   Local epochs  : {LOCAL_EPOCHS} per round')
print(f'   Algorithm     : FedAvg')
print()

# Initialise global model
global_model = build_federated_mlp(FL_INPUT_DIM, FL_N_CLASSES)
global_weights = global_model.get_weights()

# Prepare client datasets (one per zone)
client_datasets = []
for cid in range(N_CLIENTS):
    idx  = df_fe['cluster'] == cid
    X_cl = X_scaled[idx]
    y_cl = y[idx]
    X_tr_cl, X_te_cl, y_tr_cl, y_te_cl = train_test_split(
        X_cl, y_cl, test_size=0.2, random_state=SEED)
    client_datasets.append((X_tr_cl, X_te_cl, y_tr_cl, y_te_cl))

# Communication rounds
round_accuracies = []
for rnd in range(N_ROUNDS):
    client_weights_list = []
    client_sizes        = []

    # ---- Each client trains locally ----
    for cid in range(N_CLIENTS):
        X_tr_cl, X_te_cl, y_tr_cl, y_te_cl = client_datasets[cid]

        # Local model starts from global weights
        local_model = build_federated_mlp(FL_INPUT_DIM, FL_N_CLASSES)
        local_model.set_weights(global_weights)

        # Train locally for LOCAL_EPOCHS (no data leaves this 'device')
        local_model.fit(X_tr_cl, y_tr_cl,
                        epochs=LOCAL_EPOCHS, batch_size=32,
                        verbose=0, validation_split=0.1)

        client_weights_list.append(local_model.get_weights())
        client_sizes.append(len(X_tr_cl))

    # ---- Server aggregates (FedAvg) ----
    global_weights = fedavg_aggregate(
        global_weights, client_weights_list, client_sizes)
    global_model.set_weights(global_weights)

    # ---- Evaluate global model on all clients ----
    accs = []
    for cid in range(N_CLIENTS):
        _, X_te_cl, _, y_te_cl = client_datasets[cid]
        preds = np.argmax(global_model.predict(X_te_cl, verbose=0), axis=1)
        accs.append(accuracy_score(y_te_cl, preds))
    mean_acc = np.mean(accs)
    round_accuracies.append(mean_acc)
    print(f'  Round {rnd+1}/{N_ROUNDS} — Global model accuracy: {mean_acc:.4f}  '
          f'(per-client: {[f"{a:.3f}" for a in accs]})')

# ---- Plot federated learning convergence ----
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, N_ROUNDS+1), round_accuracies, 'bo-',
        linewidth=2, markersize=10)
for i, acc in enumerate(round_accuracies):
    ax.annotate(f'{acc:.3f}', (i+1, acc), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=10)
ax.set_title('FedAvg Convergence — Global Model Accuracy per Communication Round',
             fontweight='bold')
ax.set_xlabel('Communication Round')
ax.set_ylabel('Mean Accuracy across Clients')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('federated_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅ Federated Learning simulation complete.')
print(f'   Final global model accuracy: {round_accuracies[-1]:.4f}')
print(f'   Key insight: model improves each round without raw data sharing.')
print(f'\n   Production notes:')
print(f'   • Replace local_model.fit() with on-device training (TFLite/CoreML)')
print(f'   • Add DP noise: weights += Gaussian(0, σ) before sending to server')
print(f'   • Use secure aggregation (encrypted weight upload) for stronger privacy')
print(f'   • FedProx variant handles heterogeneous client data better than FedAvg')


---
# CELL 16 — Complete Agent: All Modules Integrated

In [ ]:
# ============================================================
# CELL 16 — Ultimate FarmRecommendationAgent
#
# Integrates every module:
#   ✅ HDBSCAN zone assignment
#   ✅ Optuna-tuned + calibrated ML models
#   ✅ MC-Dropout uncertainty quantification
#   ✅ SHAP per-farm feature explanation
#   ✅ Yield prediction (tons/ha)
#   ✅ Multi-objective Pareto optimisation
#   ✅ ChromaDB RAG (similar farm retrieval)
#   ✅ Llama 3.1 via Groq (natural language + JSON output)
#   ✅ Multi-turn conversational memory
#   ✅ Graceful fallback (ML-only if LLM unavailable)
# ============================================================

SYSTEM_PROMPT = """
You are an expert agricultural advisor AI in a precision farming app.
You receive structured data from an ML ensemble including crop predictions,
yield forecasts, SHAP explanations, and similar historical farms.
Generate a rich, actionable recommendation as ONLY valid JSON — no markdown.

OUTPUT FORMAT (strict JSON):
{
  "primary_recommendation": {
    "crop": "crop name",
    "confidence": 87.3,
    "confidence_interval": [82.1, 91.4],
    "predicted_yield": "3.8 tons/ha",
    "one_line_reason": "Why this crop fits this specific farm in one sentence",
    "soil_analysis": "2-3 sentences on N/P/K/pH suitability",
    "climate_analysis": "2-3 sentences on temperature/humidity/rainfall",
    "satellite_insight": "1-2 sentences interpreting NDVI time-series pattern",
    "top_shap_drivers": ["Feature1: +0.34 (pushes toward this crop)", "Feature2: +0.22"],
    "action_steps": ["Step 1", "Step 2", "Step 3"],
    "risk_alerts": ["Alert if any"],
    "best_planting_window": "Month range",
    "uncertainty_flag": false
  },
  "pareto_alternatives": [
    {"crop": "crop2", "yield": "2.1 t/ha", "water": "low",
     "carbon": "low", "reason": "Best if water is scarce"},
    {"crop": "crop3", "yield": "4.2 t/ha", "water": "high",
     "carbon": "medium", "reason": "Highest yield but water-intensive"}
  ],
  "agronomic_zone": "One-sentence zone description",
  "sustainability_tip": "One eco-friendly tip",
  "similar_farms_insight": "What the 5 similar farms suggest",
  "follow_up_answer": null
}
"""

def compute_s2_single(N, P, K, humidity, rainfall):
    N_s = np.clip(N/140,0.05,0.95); P_s = np.clip(P/145,0.05,0.95)
    hum_s = np.clip(humidity/100,0.05,0.95)
    rain_s = np.clip(rainfall/300,0.05,0.95)
    NIR = 0.4*N_s+0.3*hum_s+0.3*rain_s
    Red = 0.6*(1-N_s)+0.4*(1-P_s)
    Green = 0.5*hum_s+0.5*rain_s
    Blue = 0.2*P_s+0.08; eps=1e-8
    return (float(np.clip((NIR-Red)/(NIR+Red+eps),-1,1)),
            float(np.clip((Green-NIR)/(Green+NIR+eps),-1,1)),
            float(np.clip(((NIR-Red)/(NIR+Red+0.5+eps))*1.5,-1,1)),
            float(np.clip(2.5*(NIR-Red)/(NIR+6*Red-7.5*Blue+1+eps),-1,1)))

def call_llama(messages, max_tokens=1500, retries=2):
    for attempt in range(retries):
        try:
            resp = groq_client.chat.completions.create(
                model='llama-3.1-8b-instant', messages=messages,
                max_tokens=max_tokens, temperature=0.3, top_p=0.9)
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r'^```json\s*','',raw)
            raw = re.sub(r'^```\s*','',raw)
            raw = re.sub(r'\s*```$','',raw)
            return json.loads(raw)
        except Exception:
            if attempt < retries-1: time.sleep(1)
    return None


class UltimateFarmAgent:
    """
    Production-ready intelligent farm recommendation agent.
    Combines HDBSCAN zones, Optuna-tuned calibrated models,
    MC-Dropout uncertainty, SHAP explanations, yield regression,
    Pareto multi-objective optimisation, ChromaDB RAG, and
    Llama 3.1 natural language generation into a single interface.
    """

    def __init__(self):
        self.conversation_history = []
        self.last_result          = None

    def _get_uncertainty(self, cid, x_local, le_local, n_passes=50):
        """
        MC-Dropout uncertainty: run 50 stochastic forward passes.
        Returns (mean_proba, uncertainty_score).
        uncertainty_score = mean std across all classes (0=certain, 1=very uncertain)
        """
        cnn_preds, cnn_std = mc_dropout_predict(
            cluster_models[cid]['CNN1D'], x_local, n_passes=n_passes, is_cnn=True)
        mlp_preds, mlp_std = mc_dropout_predict(
            cluster_models[cid]['MLP'],   x_local, n_passes=n_passes, is_cnn=False)

        # Average uncertainty across DL models
        uncertainty_score = float(np.mean([cnn_std.mean(), mlp_std.mean()]))
        avg_dl_proba      = (cnn_preds + mlp_preds) / 2

        return avg_dl_proba[0], uncertainty_score

    def _get_shap_explanation(self, cid, x_scaled, top_n=5):
        """
        Compute SHAP values for a single farm and return top drivers.
        Uses the RF base estimator from the CalibratedClassifierCV wrapper.
        """
        try:
            rf_base = cluster_models[cid]['RF'].calibrated_classifiers_[0].estimator
            explainer = shap.TreeExplainer(rf_base)
            shap_vals = explainer.shap_values(x_scaled)

            # For multi-class: take the class with highest probability
            if isinstance(shap_vals, list):
                sv = shap_vals[0]
            else:
                sv = shap_vals

            sv_row = sv[0] if len(sv.shape) > 1 else sv

            # Build sorted feature contribution list
            contrib = list(zip(FEATURE_COLS, sv_row))
            contrib.sort(key=lambda t: abs(t[1]), reverse=True)
            top = [f"{feat}: {'+' if val>0 else ''}{val:.3f}"
                   for feat, val in contrib[:top_n]]
            return top
        except Exception:
            return ['SHAP unavailable for this zone']

    def recommend(self, N, P, K, temperature, humidity, ph, rainfall,
                  ndvi=None, ndwi=None, savi=None, evi=None,
                  use_llm=True, verbose=True):
        """
        Full recommendation pipeline with all enhancements.
        """
        # ---- 1. Sentinel-2 indices ----
        if ndvi is None:
            NDVI, NDWI, SAVI, EVI = compute_s2_single(N,P,K,humidity,rainfall)
        else:
            NDVI, NDWI, SAVI, EVI = ndvi, ndwi, savi, evi

        # ---- 2. Feature engineering ----
        farm_data = dict(N=N,P=P,K=K,temperature=temperature,
                         humidity=humidity,ph=ph,rainfall=rainfall,
                         NDVI=NDVI,NDWI=NDWI,SAVI=SAVI,EVI=EVI)

        x_raw    = engineer_single_full(farm_data)
        x_scaled_f = scaler.transform(x_raw.reshape(1,-1))

        # ---- 3. HDBSCAN zone assignment ----
        x_latent = encoder.predict(x_scaled_f, verbose=0)
        # Use approximate_predict for inference on new points
        try:
            labels, strengths = hdbscan.approximate_predict(clusterer, x_latent)
            cid = int(labels[0])
            if cid == -1:
                # Outlier: assign to zone with highest membership strength
                soft = hdbscan.all_points_membership_vectors(clusterer)
                cid  = int(np.argmax(strengths))
        except Exception:
            # Fallback: nearest centroid
            dists = [np.linalg.norm(x_latent -
                     X_latent[df_fe['cluster']==c].mean(axis=0))
                     for c in range(OPTIMAL_K)]
            cid = int(np.argmin(dists))

        le_local = cluster_encoders[cid]
        x_local  = cluster_scalers[cid].transform(x_scaled_f)

        # ---- 4. ML predictions (calibrated probabilities) ----
        all_probs  = []
        model_votes = {}
        for name in ['RF','XGB','GBM','Ensemble']:
            p = cluster_models[cid][name].predict_proba(x_scaled_f)[0]
            all_probs.append(p)
            top_local = le_local.inverse_transform([np.argmax(p)])[0]
            model_votes[name] = le.inverse_transform([top_local])[0]

        # ---- 5. MC-Dropout uncertainty ----
        dl_proba, uncertainty = self._get_uncertainty(cid, x_local, le_local)
        all_probs.append(dl_proba)
        uncertainty_flag = uncertainty > 0.15  # flag if std > 15%

        # ---- 6. Meta-ensemble average ----
        avg_proba    = np.mean(all_probs, axis=0)
        top3_idx     = np.argsort(avg_proba)[::-1][:3]
        top3_local   = le_local.inverse_transform(top3_idx)
        top3_global  = le.inverse_transform(top3_local)
        top3_conf    = [round(float(avg_proba[i])*100, 2) for i in top3_idx]

        # Confidence interval from MC-Dropout std
        dl_std_top = float(dl_proba[top3_idx[0]])
        conf_lo = max(0, top3_conf[0] - uncertainty*100*2)
        conf_hi = min(100, top3_conf[0] + uncertainty*100*2)

        farm_data['ndvi_mean']      = farm_data['NDVI']
        farm_data['ndvi_max']       = farm_data['NDVI'] + 0.1
        farm_data['ndvi_min']       = farm_data['NDVI'] - 0.1
        farm_data['ndvi_std']       = 0.05
        farm_data['ndvi_amplitude'] = 0.2
        farm_data['water_balance']  = rainfall - 0.3*temperature*humidity
        farm_data['NPK_sum']        = N + P + K

        # ---- 7. Yield prediction ----
        try:
            crop_encoded_y = le_yield.transform([top3_global[0]])[0]
            x_yield_in = np.append(x_scaled_f, crop_encoded_y).reshape(1,-1)
            pred_yield  = round(float(yield_model.predict(x_yield_in)[0]), 2)
        except Exception:
            pred_yield = None

        # ---- 8. SHAP explanation ----
        shap_drivers = self._get_shap_explanation(cid, x_scaled_f)

        # ---- 9. Multi-objective Pareto ----
        ml_result_for_pareto = {
            'cluster_id' : cid,
            'top_crops'  : top3_global.tolist(),
            'confidences': top3_conf
        }
        try:
            df_all_cands, df_pareto = get_pareto_crops(
                ml_result_for_pareto, farm_data, top_n=3)
        except Exception:
            df_pareto = None

        # ---- 10. ChromaDB RAG retrieval ----
        similar_farms_text = retrieve_similar_farms(farm_data, n_results=5)

        # ---- 11. Build LLM context ----
        pareto_text = ''
        if df_pareto is not None and len(df_pareto) > 0:
            pareto_text = 'PARETO-OPTIMAL ALTERNATIVES:\n'
            for _, r in df_pareto.iterrows():
                pareto_text += (
                    f"  {r['crop']}: yield={r['yield']:.1f}t/ha "
                    f"water={r['water_usage']:.0f}mm/yr "
                    f"carbon={r['carbon']:.0f}kgCO2/t\n")

        context = f"""
FARM SENSOR DATA:
  N={N} mg/kg  P={P} mg/kg  K={K} mg/kg
  Temperature={temperature}°C  Humidity={humidity}%  pH={ph}  Rainfall={rainfall}mm

SENTINEL-2 INDICES:
  NDVI={NDVI:.3f}  NDWI={NDWI:.3f}  SAVI={SAVI:.3f}  EVI={EVI:.3f}

ML ENSEMBLE (calibrated probabilities):
  #1 {top3_global[0]}: {top3_conf[0]:.1f}% (CI: [{conf_lo:.1f}%, {conf_hi:.1f}%])
  #2 {top3_global[1]}: {top3_conf[1]:.1f}%
  #3 {top3_global[2]}: {top3_conf[2]:.1f}%
  Uncertainty score: {uncertainty:.3f} {'⚠️ HIGH — flag for agronomist review' if uncertainty_flag else '✅ LOW'}
  Model votes: {model_votes}

YIELD PREDICTION: {pred_yield} tons/ha (for {top3_global[0]})

SHAP TOP DRIVERS (why {top3_global[0]} was recommended):
  {chr(10).join(shap_drivers)}

{pareto_text}

AGRO-ECOLOGICAL ZONE {cid}:
  Size={CLUSTER_PROFILES[cid]['size']} farms
  Dominant crops={CLUSTER_PROFILES[cid]['dominant_crops']}
  Avg N/P/K={CLUSTER_PROFILES[cid]['mean_N']}/{CLUSTER_PROFILES[cid]['mean_P']}/{CLUSTER_PROFILES[cid]['mean_K']}
  Avg NDVI={CLUSTER_PROFILES[cid]['mean_NDVI']}
  Avg NDVI amplitude={CLUSTER_PROFILES[cid]['mean_ndvi_amplitude']}

{similar_farms_text}

Generate the complete JSON recommendation.
""".strip()

        # ---- 12. Call Llama 3.1 ----
        llm_result = None
        if use_llm:
            self.conversation_history = [
                {'role': 'system',  'content': SYSTEM_PROMPT},
                {'role': 'user',    'content': context}
            ]
            llm_result = call_llama(self.conversation_history)
            if llm_result:
                self.conversation_history.append({
                    'role': 'assistant',
                    'content': json.dumps(llm_result)
                })

        # ---- 13. Build final structured response ----
        response = {
            'status'        : 'success',
            'farm_input'    : farm_data,
            'zone_id'       : cid,
            'zone_profile'  : CLUSTER_PROFILES[cid],
            'ml_engine'     : {
                'top_crops'          : top3_global.tolist(),
                'confidences'        : top3_conf,
                'confidence_interval': [round(conf_lo,1), round(conf_hi,1)],
                'model_votes'        : model_votes,
                'uncertainty_score'  : round(uncertainty, 4),
                'uncertainty_flag'   : bool(uncertainty_flag),
                'predicted_yield_tha': pred_yield,
                'shap_top_drivers'   : shap_drivers,
            },
            'pareto_front'      : df_pareto.to_dict('records') if df_pareto is not None else [],
            'llm_recommendation': llm_result,
            'llm_available'     : llm_result is not None,
        }
        self.last_result = response

        if verbose:
            self._print(response)
        return response

    def ask(self, question, verbose=True):
        """Multi-turn follow-up with full context memory."""
        if not self.conversation_history:
            return {'error': 'Call recommend() first.'}
        self.conversation_history.append({'role':'user','content':
            f'Farmer asks: "{question}". Answer specifically for this farm. '
            f'Return same JSON structure with follow_up_answer populated.'})
        result = call_llama(self.conversation_history, max_tokens=600)
        if result:
            self.conversation_history.append({'role':'assistant',
                                              'content':json.dumps(result)})
            if verbose:
                print(f'\n💬 Q: {question}')
                print(f'🌱 A: {result.get("follow_up_answer","-")}')
        return result

    def to_json(self, result):
        """Return app-ready JSON string."""
        return json.dumps(result, indent=2, ensure_ascii=False, default=str)

    def _print(self, r):
        ml = r['ml_engine']
        print('\n' + '='*68)
        print('  🌾  ULTIMATE FARM RECOMMENDATION REPORT')
        print('='*68)
        fi = r['farm_input']
        print(f'\n📊 Farm: N={fi["N"]} P={fi["P"]} K={fi["K"]} '
              f'Temp={fi["temperature"]}°C Hum={fi["humidity"]}% '
              f'pH={fi["ph"]} Rain={fi["rainfall"]}mm')
        print(f'📡 Sentinel-2: NDVI={fi["NDVI"]:.3f} NDWI={fi["NDWI"]:.3f}')
        print(f'🗺️  Zone: {r["zone_id"]} | '
              f'Dominant: {r["zone_profile"]["dominant_crops"]}')
        print(f'\n🏆 ML Ensemble (calibrated):')
        for crop, conf in zip(ml['top_crops'], ml['confidences']):
            print(f'   {crop:<18} {conf:.1f}%')
        print(f'   CI: [{ml["confidence_interval"][0]}%, '
              f'{ml["confidence_interval"][1]}%]')
        print(f'   Uncertainty: {ml["uncertainty_score"]:.4f} '
              f'{"⚠️ HIGH" if ml["uncertainty_flag"] else "✅ LOW"}')
        print(f'   Predicted yield: {ml["predicted_yield_tha"]} t/ha')
        print(f'\n🔍 SHAP top drivers:')
        for d in ml['shap_top_drivers'][:3]:
            print(f'   {d}')
        pf = r['pareto_front']
        if pf:
            print(f'\n⚖️  Pareto-optimal alternatives:')
            for p in pf[:3]:
                print(f'   {p["crop"]:<15} '
                      f'yield={p["yield"]:.1f}t/ha '
                      f'water={p["water_usage"]:.0f}mm '
                      f'carbon={p["carbon"]:.0f}')
        llm = r['llm_recommendation']
        if llm:
            pr = llm.get('primary_recommendation', {})
            print(f'\n🦙 Llama 3.1: {pr.get("one_line_reason","-")}')
            for step in pr.get('action_steps', []):
                print(f'   • {step}')
        print('='*68)


# Instantiate
ultimate_agent = UltimateFarmAgent()
print('✅ UltimateFarmAgent ready — all modules integrated.')


---
# CELL 17 — Test the Complete Agent

In [ ]:
# ============================================================
# CELL 17 — Test the complete agent with all enhancements active
# ============================================================

# Test 1: Tropical humid farm
print('TEST 1 — Tropical humid paddy farm')
result1 = ultimate_agent.recommend(
    N=90, P=42, K=43, temperature=21.0,
    humidity=82.0, ph=6.5, rainfall=210.0
)


In [ ]:
# Test 2: Semi-arid drought-tolerant scenario
print('\nTEST 2 — Semi-arid farm')
agent2 = UltimateFarmAgent()
result2 = agent2.recommend(
    N=20, P=15, K=30, temperature=28.0,
    humidity=40.0, ph=7.2, rainfall=55.0
)


In [ ]:
# Multi-turn follow-up
ultimate_agent.ask('How much NPK fertiliser should I apply and when?')


In [ ]:
ultimate_agent.ask('What pests should I watch for and how do I prevent them organically?')


In [ ]:
# Print app-ready JSON
print('\n📦 APP-READY JSON OUTPUT:')
print(ultimate_agent.to_json(result1)[:2000], '...\n[truncated]')


---
# CELL 18 — Flask REST API + Hugging Face Spaces Deployment

## Why Hugging Face Spaces instead of ngrok?
- ngrok free tier: URL changes every 8 hours, Colab must stay open
- Hugging Face Spaces: permanent URL, free, no timeout, supports Gradio + Flask
- Your integration colleague gets a stable API URL that survives notebook shutdowns


In [ ]:
# ============================================================
# CELL 18 — Flask REST API (same endpoints as before, enhanced response)
# For permanent deployment: copy this to a Hugging Face Space
# following the guide below.
# ============================================================
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

app = Flask(__name__)
CORS(app)

api_agent = UltimateFarmAgent()

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status'  : 'ok',
        'model'   : 'llama-3.1-8b-instant',
        'zones'   : OPTIMAL_K,
        'features': len(FEATURE_COLS),
        'modules' : ['HDBSCAN','SHAP','Optuna','Calibration',
                     'MC-Dropout','YieldRegression','Pareto','RAG','Llama3']
    })

@app.route('/recommend', methods=['POST'])
def recommend_endpoint():
    global api_agent
    data = request.get_json(force=True)
    required = ['N','P','K','temperature','humidity','ph','rainfall']
    missing  = [f for f in required if f not in data]
    if missing:
        return jsonify({'status':'error','message':f'Missing: {missing}'}), 400
    try:
        api_agent = UltimateFarmAgent()
        result = api_agent.recommend(
            N=float(data['N']), P=float(data['P']), K=float(data['K']),
            temperature=float(data['temperature']),
            humidity=float(data['humidity']),
            ph=float(data['ph']),
            rainfall=float(data['rainfall']),
            ndvi=data.get('ndvi'), ndwi=data.get('ndwi'),
            savi=data.get('savi'), evi=data.get('evi'),
            use_llm=data.get('use_llm', True),
            verbose=False
        )
        return jsonify(result)
    except Exception as e:
        return jsonify({'status':'error','message':str(e)}), 500

@app.route('/followup', methods=['POST'])
def followup_endpoint():
    data = request.get_json(force=True)
    question = data.get('question','').strip()
    if not question:
        return jsonify({'status':'error','message':'question required'}), 400
    try:
        result = api_agent.ask(question, verbose=False)
        return jsonify(result or {'status':'error','message':'LLM unavailable'})
    except Exception as e:
        return jsonify({'status':'error','message':str(e)}), 500

def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

# ---- Expose via ngrok (Colab testing) ----
from pyngrok import ngrok
NGROK_TOKEN = 'YOUR_NGROK_TOKEN'   # get free token at dashboard.ngrok.com
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000).public_url

print('='*62)
print('  🚀 ULTIMATE FARM API IS LIVE')
print('='*62)
print(f'  Temporary (Colab) : {public_url}')
print(f'  Health            : {public_url}/health')
print(f'  Recommend         : POST {public_url}/recommend')
print(f'  Follow-up         : POST {public_url}/followup')
print()
print('  HUGGING FACE SPACES PERMANENT DEPLOYMENT:')
print('  1. Go to https://huggingface.co/spaces → New Space')
print('  2. SDK: Gradio  |  Visibility: Public')
print('  3. Upload: app.py (Flask code above) + requirements.txt')
print('  4. requirements.txt: flask flask-cors groq chromadb')
print('     sentence-transformers xgboost scikit-learn tensorflow')
print('     hdbscan shap optuna pymoo numpy pandas')
print('  5. Add GROQ_API_KEY to Space Secrets (Settings → Secrets)')
print('  6. Your permanent URL: https://your-username-space-name.hf.space')
print('='*62)
